<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_85_gans_autoencoders_vae.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🎭 **Module 85 — GANs · Autoencoders · VAE** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 11 · Foundations Backfill


# Module 85 — GANs · Autoencoders · VAE

> M21 covered **diffusion**, the dominant generative model in 2026. But for the decade before that — and **still under the hood today** — the field was carved up between **autoencoders (AE)**, **variational autoencoders (VAE)**, **generative adversarial networks (GAN)**, and their discrete cousin **VQ-VAE**. Stable Diffusion's latent space *is a VAE*. DALL-E 1, MUSE, AudioCraft, and most audio tokenisers *are VQ-VAEs*. StyleGAN is still the SOTA for human-face editing. This module is that depth pass.

### What you'll cover
1. **The generative-model taxonomy** — explicit density vs implicit vs score-based
2. **Autoencoders** — the bottleneck idea + denoising / sparse / contractive variants
3. **Variational AE (VAE)** — the probabilistic upgrade + KL term + **reparameterisation trick**
4. **GAN** — adversarial training; the minimax game; mode collapse
5. **DCGAN · WGAN · WGAN-GP · StyleGAN** — the GAN lineage
6. **VQ-VAE** — discrete codes; the secret behind DALL-E 1, MUSE, AudioCraft, Encodec
7. **Where each still matters in 2026** — and where they lost to diffusion (M21 / M86)
8. Code: a tiny AE + VAE + GAN you can train on MNIST in 5 minutes


## 1 · The generative-model taxonomy

Every generative model wants to learn `p(x)` (or sample from it). The taxonomy:

```
                                       Generative models
                                              │
        ┌─────────────────────────────────────┼─────────────────────────────────────┐
        ▼                                     ▼                                     ▼
    EXPLICIT density                    IMPLICIT density                     SCORE-based
    (write p(x) or a bound)              (learn a sampler, no p(x))           (learn ∇log p(x))
        │                                     │                                     │
        ├── Autoregressive                    ├── GANs                              └── Diffusion (M21, M86)
        │   (PixelCNN, GPT — M19)            │   (DCGAN, StyleGAN, …)                Flow-matching
        ├── Normalizing flows                                                        Score-matching
        │   (RealNVP, Glow)
        ├── Variational AE
        │   (VAE, β-VAE, NVAE)
        └── VQ-VAE
            (discrete latents)
```

Four trade-offs you'll keep meeting:

| Family | Pros | Cons |
|---|---|---|
| **Autoregressive** | exact likelihood, easy training | slow sequential sampling |
| **Normalizing flow** | exact likelihood, fast sampling | architecture constraints (bijective) |
| **VAE** | fast sampling, smooth latent | blurry samples; ELBO is a *bound*, not exact |
| **GAN** | sharp samples | unstable training, mode collapse, no likelihood |
| **VQ-VAE + autoregressive prior** | discrete tokens, scalable | two-stage training |
| **Diffusion** (M21, M86) | best sample quality | slow multi-step sampling (unless distilled) |

The 2026 verdict: **diffusion** dominates images / video / audio quality; **VAEs live on as the encoder/decoder *inside* diffusion**; **VQ-VAEs live on as audio + image tokenisers for autoregressive models** like MUSE, AudioCraft, MagViT; **GANs lost the main race but still win specific real-time low-latency niches** (StyleGAN, neural face renderers).

## 2 · Autoencoders — the bottleneck

The simplest generative-ish architecture: an **encoder** compresses input `x` to a low-dimensional **code** `z`; a **decoder** reconstructs `x` from `z`. Loss = reconstruction error.

```
   x  ──► [ encoder ]  ──► z  ──► [ decoder ]  ──► x̂      loss = ||x − x̂||²
                            │
                            └── small bottleneck dim (e.g. 32) forces the network
                                to learn a compressed, useful representation
```

What you get:
- **Dimensionality reduction** — like nonlinear PCA.
- **Denoising** — train with corrupted input, clean target → **denoising autoencoder (DAE)** (Vincent 2008), the ancestor of diffusion.
- **Anomaly detection** — high reconstruction error = "unusual."
- **Pretraining** — features from `z` transfer to downstream tasks.

Three useful variants:

| Variant | Trick |
|---|---|
| **Denoising AE** | input is `x + noise`, target is clean `x` |
| **Sparse AE** | add L1 penalty on `z` → forces few active units |
| **Contractive AE** | penalise the Frobenius norm of the encoder's Jacobian → robust features |
| **Masked AE (MAE)** | mask 75% of input patches; predict the rest. Used in **MAE** (He et al. 2021) — the self-supervised pretraining recipe for ViT |

In [ ]:
import torch, torch.nn as nn

class AutoEncoder(nn.Module):
    def __init__(self, in_dim=784, z_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(in_dim, 256), nn.ReLU(),
                                      nn.Linear(256, z_dim))
        self.decoder = nn.Sequential(nn.Linear(z_dim, 256), nn.ReLU(),
                                      nn.Linear(256, in_dim), nn.Sigmoid())
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

# Training is just MSE / BCE
ae = AutoEncoder()
x = torch.rand(64, 784)             # batch of MNIST-flattened images
x_hat, z = ae(x)
loss = nn.functional.binary_cross_entropy(x_hat, x)
print(loss.item(), "  z shape:", z.shape)

**The AE limitation that motivated VAE:** the latent space `z` has **no probability structure**. You can't sample new `z` and decode it — interpolation produces garbage, novel `z` is undefined. That's what VAE fixes.

## 3 · VAE — the probabilistic upgrade

Kingma & Welling (2013). Instead of a deterministic `z`, the encoder outputs a **distribution** `q(z|x) = N(μ(x), σ(x)²)`. Sample `z` from it, decode, reconstruct.

The loss has **two terms**:

$$\mathcal{L}_{VAE} = \underbrace{\mathbb{E}_{q(z|x)}[-\log p(x|z)]}_{\text{reconstruction}} + \underbrace{\beta \cdot D_{KL}(q(z|x) \,\|\, p(z))}_{\text{regularises latent toward }\mathcal{N}(0, I)}$$

The KL term forces the encoder's distribution toward a standard Normal **prior**. After training you can sample `z ~ N(0, I)` and decode to get a new sample. **That's a generative model.**

### The reparameterisation trick

You can't backprop through a sampling operation. Trick:

$$z = \mu(x) + \sigma(x) \cdot \epsilon, \qquad \epsilon \sim \mathcal{N}(0, I)$$

The randomness lives in `ε`, which has no parameters. Gradients flow through `μ(x)` and `σ(x)` cleanly. This trick is so fundamental it appears in Diffusion (M21 / M86), DPO (M62), and GRPO (M67).

In [ ]:
class VAE(nn.Module):
    def __init__(self, in_dim=784, z_dim=32):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(in_dim, 256), nn.ReLU())
        self.mu_head     = nn.Linear(256, z_dim)
        self.logvar_head = nn.Linear(256, z_dim)
        self.dec = nn.Sequential(nn.Linear(z_dim, 256), nn.ReLU(),
                                  nn.Linear(256, in_dim), nn.Sigmoid())
    def encode(self, x):
        h = self.enc(x)
        return self.mu_head(h), self.logvar_head(h)
    def reparameterise(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + std * eps                              # ← the trick
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterise(mu, logvar)
        x_hat = self.dec(z)
        return x_hat, mu, logvar

def vae_loss(x_hat, x, mu, logvar, beta=1.0):
    recon = nn.functional.binary_cross_entropy(x_hat, x, reduction="sum")
    kl    = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + beta * kl

vae = VAE()
x = torch.rand(64, 784)
x_hat, mu, lv = vae(x)
print("loss:", vae_loss(x_hat, x, mu, lv).item())

### β-VAE and the disentanglement story
Higgins et al. (2017) added a `β` knob in front of the KL term. **High β** forces a tighter prior → individual latent dimensions become more interpretable (one captures rotation, another scale, …). **The trade-off:** higher β = blurrier reconstructions. β-VAE was a major step toward "controllable" generation.

### Where VAEs survive in 2026
- **Stable Diffusion's first stage** is a VAE. It compresses 512×512 images to 64×64×4 *latent* space; diffusion then runs there. Without the VAE, diffusion on full pixels would be ~32× more expensive.
- **VQ-VAE** (next section) — discrete-latent variant powering many audio + image tokenisers.
- **Tabular VAE** (TVAE, CTGAN's sibling) — synthetic structured data for tabular ML.

## 4 · GAN — the adversarial game

Goodfellow et al. (2014). Two networks play a minimax game:

```
   Generator G : noise z ~ N(0,I)  ──► fake sample G(z)
                                            │
                                            ▼
   Discriminator D : sample x ──► P(x is real)  ∈ [0, 1]

   D's goal: tell real from fake.
   G's goal: fool D.
```

The objective:

$$\min_G \max_D \mathbb{E}_{x \sim p_{data}}[\log D(x)] + \mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))]$$

Trained alternately: one step of `D` (with real + fake), one step of `G` (trying to make D's "fake" score high).

### What goes wrong
GANs are notoriously hard to train. Five canonical failures:

| Failure | Symptom | Fix |
|---|---|---|
| **Mode collapse** | G produces 1-3 distinct samples regardless of `z` | minibatch discrimination, feature matching, WGAN |
| **Vanishing G gradient** | once D is "too good," G gets no signal | non-saturating loss (D=binary CE instead of log) |
| **Training oscillation** | loss bounces; quality regresses | smaller LR, TTUR (different LR for G vs D) |
| **No likelihood** | can't tell whether you're making progress | Inception Score, FID, KID, precision/recall |
| **Hyperparameter brittleness** | a tiny LR change destroys training | use known recipes — DCGAN, StyleGAN |

> 🟡 **The GAN truth**: every successful GAN paper is more about **stabilising training** than the architecture itself.

In [ ]:
# A tiny DCGAN sketch — generator + discriminator. Production-grade DCGAN
# adds BatchNorm + transpose-conv + LeakyReLU on D; this is the minimal idea.
class Generator(nn.Module):
    def __init__(self, z_dim=64, out_dim=784):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, 256), nn.ReLU(),
            nn.Linear(256, 512),   nn.ReLU(),
            nn.Linear(512, out_dim), nn.Tanh(),   # outputs in [-1, 1]
        )
    def forward(self, z): return self.net(z)

class Discriminator(nn.Module):
    def __init__(self, in_dim=784):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 512), nn.LeakyReLU(0.2),
            nn.Linear(512, 256),    nn.LeakyReLU(0.2),
            nn.Linear(256, 1),      nn.Sigmoid(),
        )
    def forward(self, x): return self.net(x)

# Training loop sketch
G, D = Generator(), Discriminator()
opt_G = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

# (training loop omitted for brevity — alternate D, G updates with BCE loss)
print("G params:", sum(p.numel() for p in G.parameters()))
print("D params:", sum(p.numel() for p in D.parameters()))

## 5 · The GAN lineage

| Architecture | Year | Key insight |
|---|---|---|
| **GAN** | 2014 | the original minimax setup |
| **DCGAN** | 2015 | transposed convs in G, strided convs in D, BatchNorm, no FC layers → first stable GAN |
| **InfoGAN** | 2016 | maximise mutual info between code and output → disentanglement |
| **Pix2Pix** | 2017 | conditional GAN; paired image translation |
| **CycleGAN** | 2017 | **unpaired** image translation — horses ↔ zebras |
| **WGAN** | 2017 | Wasserstein distance + weight clipping → stable training |
| **WGAN-GP** | 2017 | gradient penalty instead of weight clipping → SOTA stability |
| **Progressive GAN** | 2017 | train at low res first, then progressively higher → 1024² faces |
| **StyleGAN / 2 / 3** | 2018-21 | AdaIN style injection + path-length regularisation → SOTA human-face generation; still the SOTA for face editing |
| **BigGAN** | 2018 | huge batches + class-conditioned ImageNet at 512×512 |
| **GANSpace · GAN inversion** | 2020+ | edit existing images by inverting them into a GAN's latent |
| **3D GANs** (EG3D) | 2022 | tri-plane representation for 3D-aware face / scene synthesis |

The 2022 **rise of diffusion** mostly retired GANs for unconstrained image generation. But for **real-time / low-latency / interactive editing** (face filters, video calls, photorealistic avatars, neural rendering for games), GANs still dominate because they generate in **one forward pass** rather than 20-50 diffusion steps.

## 6 · VQ-VAE — discrete latents

van den Oord et al. (2017). A VAE where the latent space is **a finite codebook** instead of continuous Gaussian. The encoder outputs a continuous vector `z_e`; we then **snap** it to the nearest codebook entry `e_k`:

```
   x ──► encoder ──► z_e (continuous)
                       │
                       ▼  argmin over codebook
                     e_k  (discrete code from codebook of size K)
                       │
                       ▼
                  decoder ──► x̂
```

Why this matters:
- **Discrete tokens** mean you can train an **autoregressive model** (Transformer M19) on the codes — the recipe behind **DALL-E 1, MUSE, MagViT, AudioCraft, Encodec, Soundstream**.
- **No KL term** — replaced by a commitment loss + EMA codebook updates.
- **Better sample quality than VAE** because the continuous-Gaussian bottleneck is gone.

### VQ-VAE-2 + multi-stage
Stacking VQ-VAEs at multiple resolutions (top: 32×32, mid: 64×64, bottom: 128×128) gives **VQ-VAE-2** — the architecture that produced near-photorealistic class-conditional faces and ImageNet samples in 2019. **Residual Vector Quantization (RVQ)** is the modern audio variant — codecs like Encodec, Soundstream, and Mimi use it to compress audio into a token stream.

### Where this lives in 2026
- **Audio tokenisers**: Encodec, Mimi, Soundstream — the audio version of "tokenize for transformers."
- **Image / video tokenisers**: MAGVIT-v2, OmniTokenizer — feed into autoregressive image / video transformers.
- **MagVIT2-style** discrete video tokens at the core of many 2024+ video models.

## 7 · Where each still matters in 2026

```
   AE              still everywhere as: pretraining, anomaly detection,
                   embedding learning, masked-AE pretraining for ViT
   Denoising AE    direct conceptual ancestor of diffusion (M21, M86)
   VAE             encoder/decoder INSIDE Stable Diffusion's latent space
                   tabular synthetic data (TVAE)
                   ML-flavoured drug / molecule design
   β-VAE / NVAE    research on disentangled representation
   VQ-VAE          the universal token for audio + image + video autoregressive
                   models — DALL-E 1, MUSE, Encodec, Mimi, MagViT, Soundstream
   GAN (DCGAN)     teaching example; still the workhorse of "fast 1-step image"
   StyleGAN        SOTA human-face editing / inversion (2018 → 2024)
   CycleGAN        unpaired domain translation; satellite ↔ map, sketch ↔ photo
   WGAN-GP         the GAN you reach for when you really need to train one
   Conditional GAN production face filters, real-time avatars, neural rendering
                   wherever 1-step inference matters more than absolute quality
   Diffusion       won everything else (M21, M86)
```

### When to pick what in a 2026 project

| You want… | Pick |
|---|---|
| Best image quality (offline) | **Diffusion** (M21 / M86) |
| Best face editing / inversion / interactive | **StyleGAN-3** |
| Real-time 1-step generation | **GAN** (often **GAN distilled from a diffusion model**) |
| Audio compression / tokenisation | **VQ-VAE / RVQ** (Encodec / Mimi) |
| Image → tokens for autoregressive | **VQ-VAE** (MAGVIT-v2 / FSQ) |
| Tabular synthetic data | **CTGAN / TVAE / CTAB-GAN+** |
| Anomaly detection | **AE** with reconstruction error |
| Self-supervised pretraining for ViT | **MAE** (Masked Autoencoder) |

## 8 · Train them on MNIST in 5 minutes

A skeleton training loop you can run yourself. The three models share the same data pipeline; only the loss and the inner update differ.

```python
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

mnist = datasets.MNIST(root="./data", train=True, download=True,
                       transform=transforms.ToTensor())
loader = DataLoader(mnist, batch_size=128, shuffle=True)
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- AE ---
ae = AutoEncoder().to(device); opt = torch.optim.Adam(ae.parameters(), lr=1e-3)
for epoch in range(5):
    for x, _ in loader:
        x = x.view(x.size(0), -1).to(device)
        x_hat, _ = ae(x)
        loss = nn.functional.binary_cross_entropy(x_hat, x)
        opt.zero_grad(); loss.backward(); opt.step()

# --- VAE ---
vae = VAE().to(device); opt = torch.optim.Adam(vae.parameters(), lr=1e-3)
for epoch in range(5):
    for x, _ in loader:
        x = x.view(x.size(0), -1).to(device)
        x_hat, mu, lv = vae(x)
        loss = vae_loss(x_hat, x, mu, lv) / x.size(0)
        opt.zero_grad(); loss.backward(); opt.step()

# Then sample from VAE:
with torch.no_grad():
    z = torch.randn(16, 32).to(device)
    samples = vae.dec(z).view(-1, 28, 28).cpu().numpy()
    # plot with matplotlib...
```

A real **DCGAN** loop is 80 lines; we leave it as an exercise. Three tips:
1. **Use the Adam(β₁=0.5, β₂=0.999) recipe** — it's the only LR config that reliably trains DCGAN.
2. **Track FID score**, not loss.
3. **Print samples every epoch** — the loss can lie; the images can't.

## ✅ Recap

- The generative-model taxonomy: **explicit density** (autoregressive, flows, VAE) · **implicit** (GAN) · **score-based** (diffusion, M21 / M86).
- **Autoencoders** compress to a bottleneck; **denoising AE** is the direct ancestor of diffusion; **MAE** is the SSL pretraining recipe for ViT (M65).
- **VAE** adds probabilistic structure to AE: Normal prior + KL term + **reparameterisation trick**. VAEs live inside Stable Diffusion's latent space.
- **GAN** is a minimax game between G and D. Mode collapse, instability, hyperparameter brittleness — every successful GAN paper is more about *stabilising training* than architecture.
- **GAN lineage**: DCGAN → WGAN → WGAN-GP → Progressive → **StyleGAN-3** (still SOTA for face editing) + **BigGAN** + conditional GANs (Pix2Pix / CycleGAN).
- **VQ-VAE** is the universal **tokeniser** behind audio (Encodec / Mimi / Soundstream), image (MAGVIT-v2), and video autoregressive models.
- 2026 verdict: **diffusion** won images; **VAE** lives as Stable Diffusion's encoder/decoder; **VQ-VAE** lives as the multimodal tokeniser; **GAN** wins the real-time / interactive niches.

Next: **M86 — Diffusion Deep Dive (SD · FLUX · schedulers · ControlNet · image-LoRA)**.
